# Inference Runtime and KV Cache


In [1]:
import sys
from pathlib import Path

ROOT = Path('/Users/montekkundan/Developer/ML/llm')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
import time
from course_tools import DEFAULT_CHAT_MESSAGES, ensure_demo_checkpoint, format_messages, prefill_prompt, decode_next_token, generate_text

LECTURE_NOTE_PATH = Path('/Users/montekkundan/Library/Mobile Documents/iCloud~md~obsidian/Documents/lectures/LLM/product/Inference Runtime and KV Cache.md')
print(LECTURE_NOTE_PATH)
bundle = ensure_demo_checkpoint(steps=40)
model = bundle['model']
tokenizer = bundle['tokenizer']
prompt = format_messages(DEFAULT_CHAT_MESSAGES, add_assistant_prompt=True)


/Users/montekkundan/Library/Mobile Documents/iCloud~md~obsidian/Documents/lectures/LLM/product/Inference Runtime and KV Cache.md


## Demo A: prefill builds the initial cache


In [2]:
prompt_ids, past_kvs = prefill_prompt(model, tokenizer, prompt)
print('prompt length:', len(prompt_ids))
print('cache shapes :', [kv[0].shape for kv in past_kvs])


prompt length: 64
cache shapes : [torch.Size([1, 4, 64, 12]), torch.Size([1, 4, 64, 12])]


## Demo B: each decode step only appends one more key and value per layer


In [3]:
current = prompt_ids[-1]
for step in range(3):
    next_id, past_kvs, _ = decode_next_token(model, current, past_kvs, temperature=0.8, top_k=8)
    print({'step': step, 'token': tokenizer.decode([next_id]), 'cache_len': past_kvs[0][0].size(-2)})
    current = next_id


{'step': 0, 'token': 't', 'cache_len': 64}
{'step': 1, 'token': 'n', 'cache_len': 64}
{'step': 2, 'token': 't', 'cache_len': 64}


## Demo C: cached decoding beats naive recomputation


In [4]:
start = time.perf_counter()
generate_text(model, tokenizer, prompt, max_new_tokens=24, temperature=0.8, top_k=8)
cached = time.perf_counter() - start
start = time.perf_counter()
text = prompt
for _ in range(24):
    text += generate_text(model, tokenizer, text, max_new_tokens=1, temperature=0.8, top_k=8)
naive = time.perf_counter() - start
print({'cached_seconds': round(cached, 4), 'naive_seconds': round(naive, 4)})


{'cached_seconds': 0.0071, 'naive_seconds': 0.0168}


## Rasbt and nanochat


**RASBT**
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch04/03_kv-cache/gpt_with_kv_cache.py
**NANOCHAT**
- https://github.com/karpathy/nanochat/blob/master/nanochat/engine.py
- https://github.com/karpathy/nanochat/blob/master/nanochat/gpt.py
